In [ ]:
!pip install -q pyspark

import sys
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("L2_StackOverflow_Report") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Пути к файлам (убедись, что они загружены в Colab)
POSTS_XML = "/content/posts_sample.xml"
LANGS_CSV = "/content/programming-languages.csv"
PARQUET_OUT = "/content/top_languages_2010_2020.parquet"

print("Spark запущен. Версия:", spark.version)

Spark запущен. Версия: 4.0.2


In [ ]:
langs_raw = spark.read.option("header", True).csv(LANGS_CSV)

langs_ref = langs_raw \
    .select(F.trim(F.lower(F.col(langs_raw.columns[0]))).alias("lang_name")) \
    .dropna() \
    .distinct() \
    .cache()

print(f"Загружено уникальных языков: {langs_ref.count()}")
langs_ref.show(5)

Загружено уникальных языков: 698
+---------+
|lang_name|
+---------+
|   ceylon|
|     hope|
|       m#|
| metafont|
|  mortran|
+---------+
only showing top 5 rows


In [ ]:
raw_xml = spark.read.text(POSTS_XML)

parsed_posts = raw_xml \
    .filter(F.col("value").contains("CreationDate=") & F.col("value").contains("Tags=")) \
    .select(
        F.regexp_extract("value", r'CreationDate="(\d{4})', 1).cast("int").alias("year"),
        F.regexp_extract("value", r'Tags="([^"]*)"', 1).alias("tags_raw")
    ) \
    .filter("year >= 2010 and year <= 2020")

tags_exploded = parsed_posts \
    .withColumn("tags_clean", F.regexp_replace("tags_raw", r"&lt;|&gt;", " ")) \
    .withColumn("tag", F.explode(F.split(F.trim("tags_clean"), r"\s+"))) \
    .select("year", F.lower(F.col("tag")).alias("tag")) \
    .filter("tag != ''")

print("Пример извлеченных тегов:")
tags_exploded.show(5)

Пример извлеченных тегов:
+----+-------------------+
|year|                tag|
+----+-------------------+
|2010|             router|
|2010|               wifi|
|2010|            netgear|
|2010|          configure|
|2010|windows-server-2008|
+----+-------------------+
only showing top 5 rows


In [ ]:
lang_usage = tags_exploded.join(
    langs_ref,
    tags_exploded.tag == langs_ref.lang_name,
    "inner"
).select("year", "lang_name")

stats = lang_usage.groupBy("year", "lang_name") \
    .agg(F.count("*").alias("count"))

window_spec = Window.partitionBy("year").orderBy(F.desc("count"), "lang_name")

top_10_report = stats \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter("rank <= 10") \
    .select("year", "lang_name", "count", "rank") \
    .orderBy("year", "rank")

print("Финальный отчет (фрагмент):")
top_10_report.show(25, truncate=False)

Финальный отчет (фрагмент):
+----+----------+-----+----+
|year|lang_name |count|rank|
+----+----------+-----+----+
|2010|php       |109  |1   |
|2010|bash      |39   |2   |
|2010|java      |26   |3   |
|2010|powershell|17   |4   |
|2010|python    |16   |5   |
|2010|perl      |6    |6   |
|2010|curl      |4    |7   |
|2010|trac      |4    |8   |
|2010|chef      |3    |9   |
|2010|coldfusion|3    |10  |
|2011|php       |135  |1   |
|2011|bash      |42   |2   |
|2011|java      |28   |3   |
|2011|powershell|27   |4   |
|2011|python    |24   |5   |
|2011|perl      |13   |6   |
|2011|ruby      |10   |7   |
|2011|sas       |8    |8   |
|2011|io        |7    |9   |
|2011|chef      |6    |10  |
|2012|php       |137  |1   |
|2012|bash      |52   |2   |
|2012|powershell|30   |3   |
|2012|java      |23   |4   |
|2012|python    |22   |5   |
+----+----------+-----+----+
only showing top 25 rows


In [ ]:
import shutil
from google.colab import files

top_10_report.write.mode("overwrite").parquet(PARQUET_OUT)
print(f"Данные сохранены в {PARQUET_OUT}")

pivot_report = top_10_report.groupBy("year").pivot("rank").agg(F.first("lang_name"))
pivot_report.orderBy("year").show(11)

shutil.make_archive("report_parquet", 'zip', PARQUET_OUT)

files.download("report_parquet.zip")

spark.stop()

Данные сохранены в /content/top_languages_2010_2020.parquet
+----+---+----+----------+----------+------+----+----+----+----+----------+
|year|  1|   2|         3|         4|     5|   6|   7|   8|   9|        10|
+----+---+----+----------+----------+------+----+----+----+----+----------+
|2010|php|bash|      java|powershell|python|perl|curl|trac|chef|coldfusion|
|2011|php|bash|      java|powershell|python|perl|ruby| sas|  io|      chef|
|2012|php|bash|powershell|      java|python|perl|ruby| awk|chef|coldfusion|
|2013|php|bash|      java|powershell|  chef|perl|sasl|curl|  io|    python|
+----+---+----+----------+----------+------+----+----+----+----+----------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>